In [1]:
import numpy as np
import os

folder = "Raw"

sample_file = os.path.join(folder, os.listdir(folder)[0])

data = np.load(sample_file)

print("Sample shape:", data.shape)

Sample shape: (2560, 27)


In [2]:
import os
import pandas as pd

data_folder = "Raw"

files = []
labels = []
subjects = []

for fname in sorted(os.listdir(data_folder)):
    if not fname.endswith(".npy"):
        continue

    path = os.path.join(data_folder, fname)

    parts = fname.split("_")
    subject = parts[2]

    if "Left" in fname:
        label = 0
    else:
        label = 1

    files.append(path)
    labels.append(label)
    subjects.append(subject)

df = pd.DataFrame({
    "file": files,
    "label": labels,
    "subject": subjects
})

print("Total samples:", len(df))
print("Total subjects:", df.subject.nunique())

df.to_csv(f"dataset_metadata_{data_folder.lower()}.csv", index=False)

Total samples: 20792
Total subjects: 87


In [3]:
from sklearn.model_selection import GroupKFold
import pandas as pd

df = pd.read_csv("dataset_metadata_raw.csv")

X = df["file"].values
y = df["label"].values
groups = df["subject"].values

gkf = GroupKFold(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    train_subjects = set(groups[train_idx])
    val_subjects = set(groups[val_idx])

    overlap = train_subjects.intersection(val_subjects)

    print("Fold", fold)
    print("Train subjects:", len(train_subjects))
    print("Val subjects:", len(val_subjects))
    print("Overlap:", len(overlap))

Fold 0
Train subjects: 69
Val subjects: 18
Overlap: 0
Fold 1
Train subjects: 69
Val subjects: 18
Overlap: 0
Fold 2
Train subjects: 70
Val subjects: 17
Overlap: 0
Fold 3
Train subjects: 70
Val subjects: 17
Overlap: 0
Fold 4
Train subjects: 70
Val subjects: 17
Overlap: 0


In [4]:
import os
import time
import random
import gc
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import (
    Layer, Input, Dense, LayerNormalization, MultiHeadAttention,
    Dropout, GlobalAveragePooling1D
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# =========================
# Reproducibility
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# =========================
# Dataset parameters
# =========================
EPOCHS = 50
N_SPLITS = 5
CHANNELS = 27
TIMEPOINTS = 2560
N_CLASSES = 2

DATASET_NAME = "Raw"  
CSV_FILE = "dataset_metadata_raw.csv"

# =========================
# Smart Hyperparameters Dictionary
# =========================
ALL_HYPERPARAMS = {
    "Raw": {
        'patch_size': 320, 'projection_dim': 128, 'num_heads': 4,
        'transformer_layers': 2, 'ff_dim': 256, 'dropout_rate': 0.34097,
        'learning_rate': 0.000192, 'batch_size': 32
    },
    "Filtered_8_30Hz": {
        'patch_size': 128, 'projection_dim': 128, 'num_heads': 8,
        'transformer_layers': 2, 'ff_dim': 128, 'dropout_rate': 0.36094,
        'learning_rate': 0.000173, 'batch_size': 32
    },
    "Filtered_8_13Hz": {
        'patch_size': 128, 'projection_dim': 64, 'num_heads': 4,
        'transformer_layers': 6, 'ff_dim': 128, 'dropout_rate': 0.37625,
        'learning_rate': 0.000185, 'batch_size': 16
    }
}

best_params = ALL_HYPERPARAMS[DATASET_NAME]
print(f"✅ Loaded Hyperparameters for {DATASET_NAME}: {best_params}")

# =========================
# Load metadata
# =========================
df = pd.read_csv(CSV_FILE)
print(f"Total samples: {len(df)}")
print(f"Total subjects: {df['subject'].nunique()}")

# =========================
# Data Generator (Adapted for ViT shape)
# =========================
class EEGGenerator_ViT(Sequence):
    def __init__(self, df, batch_size=32, shuffle=True):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indexes[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_df = self.df.iloc[batch_idx]

        X, y = [], []
        for _, row in batch_df.iterrows():
            data = np.load(row["file"])
            # ViT expects (TIMEPOINTS, CHANNELS) -> (2560, 27)
            # which is already the raw shape, so no transpose or newaxis is needed!
            X.append(data)
            y.append(row["label"])

        X = np.array(X, dtype=np.float32)
        y = tf.keras.utils.to_categorical(y, N_CLASSES)
        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

# =========================
# Transformer Layers
# =========================
class Patches(Layer):
    def __init__(self, patch_size, **kwargs):
        super(Patches, self).__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, signals):
        batch_size = tf.shape(signals)[0]
        # Reshape to (batch, num_patches, patch_size, channels)
        patches = tf.reshape(signals, [batch_size, -1, self.patch_size, tf.shape(signals)[-1]])
        return patches

    def get_config(self):
        config = super(Patches, self).get_config()
        config.update({"patch_size": self.patch_size})
        return config

class PatchEncoder(Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super(PatchEncoder, self).__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = Dense(units=projection_dim)
        
        pos_encoding_matrix = self.get_sinusoidal_position_embedding(num_patches, projection_dim)
        self.position_embedding = tf.Variable(
            pos_encoding_matrix, trainable=False, name="position_embedding"
        )

    def get_sinusoidal_position_embedding(self, sequence_length, embedding_dim):
        angle_rates = 1.0 / (10000 ** (tf.range(0, embedding_dim, 2, dtype=tf.float32) / float(embedding_dim)))
        positions = tf.range(sequence_length, dtype=tf.float32)[:, tf.newaxis]
        angle_rads = positions * angle_rates[tf.newaxis, :]
        sines = tf.math.sin(angle_rads)
        cosines = tf.math.cos(angle_rads)
        pos_encoding = tf.concat([sines, cosines], axis=-1)
        return pos_encoding

    def call(self, patch):
        batch_size = tf.shape(patch)[0]
        patch_height = patch.shape[-2]
        patch_width = patch.shape[-1]
        
        patch_flattened = tf.reshape(patch, [batch_size, self.num_patches, patch_height * patch_width])
        encoded = self.projection(patch_flattened)
        encoded += self.position_embedding 
        return encoded

    def get_config(self):
        config = super(PatchEncoder, self).get_config()
        config.update({"num_patches": self.num_patches, "projection_dim": self.projection_dim})
        return config

# =========================
# ViT-PE Model Builder
# =========================
def build_vit_pe(seq_len, input_dim, patch_size, projection_dim, num_heads, transformer_layers, ff_dim, dropout_rate, lr):
    assert seq_len % patch_size == 0, "Sequence length must be divisible by patch size."
    num_patches = seq_len // patch_size

    inputs = Input(shape=(seq_len, input_dim))
    patches = Patches(patch_size)(inputs)
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

    for _ in range(transformer_layers):
        x1 = LayerNormalization(epsilon=1e-6)(encoded_patches)
        attention_output = MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim, dropout=dropout_rate
        )(x1, x1)
        x2 = tf.keras.layers.Add()([attention_output, encoded_patches])
        x3 = LayerNormalization(epsilon=1e-6)(x2)
        x3 = Dense(ff_dim, activation="relu")(x3)
        x3 = Dropout(dropout_rate)(x3)
        x3 = Dense(projection_dim)(x3)
        encoded_patches = tf.keras.layers.Add()([x3, x2])

    representation = LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = GlobalAveragePooling1D()(representation)
    representation = Dropout(0.5)(representation)
    
    outputs = Dense(128, activation='relu')(representation)
    # ✅ Changed to Softmax for standard comparison with EEGNet
    outputs = Dense(N_CLASSES, activation='softmax')(outputs)

    model = Model(inputs=inputs, outputs=outputs)
    optimizer = Adam(learning_rate=lr)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# =========================
# Results storage
# =========================
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)
all_results = []

# =========================
# GroupKFold cross-validation
# =========================
gkf = GroupKFold(n_splits=N_SPLITS)
groups = df["subject"]

for fold, (train_idx, val_idx) in enumerate(gkf.split(df, df["label"], groups)):
    print(f"\n=============================")
    print(f"Fold {fold + 1}/{N_SPLITS}")
    print("=============================")

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    print(f"Train samples: {len(train_df)} | Val samples: {len(val_df)}")

    train_gen = EEGGenerator_ViT(train_df, best_params['batch_size'], shuffle=True)
    val_gen = EEGGenerator_ViT(val_df, best_params['batch_size'], shuffle=False)

    model = build_vit_pe(
        seq_len=TIMEPOINTS,
        input_dim=CHANNELS,
        patch_size=best_params['patch_size'],
        projection_dim=best_params['projection_dim'],
        num_heads=best_params['num_heads'],
        transformer_layers=best_params['transformer_layers'],
        ff_dim=best_params['ff_dim'],
        dropout_rate=best_params['dropout_rate'],
        lr=best_params['learning_rate']
    )

    checkpoint_path = f"models/vit_pe_{DATASET_NAME}_fold{fold + 1}.h5"
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        ModelCheckpoint(checkpoint_path, monitor="val_loss", save_best_only=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6)
    ]

    start_time = time.time()
    history = model.fit(train_gen, validation_data=val_gen,
                        epochs=EPOCHS, callbacks=callbacks, verbose=1)
    end_time = time.time()
    print(f"Training time: {end_time - start_time:.2f} seconds")

    # =========================
    # Evaluation
    # =========================
    y_true, y_pred, y_prob = [], [], []
    for X_batch, y_batch in val_gen:
        preds = model.predict(X_batch, verbose=0)
        y_true.extend(np.argmax(y_batch, axis=1))
        y_pred.extend(np.argmax(preds, axis=1))
        y_prob.extend(preds[:, 1])

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc = roc_auc_score(y_true, y_prob)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp)

    print(f"Accuracy: {acc:.3f} | F1: {f1:.3f} | ROC-AUC: {roc:.3f}")

    fold_result = {
        "dataset": DATASET_NAME,
        "fold": fold + 1,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "roc_auc": roc,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn
    }
    all_results.append(fold_result)

    del model
    tf.keras.backend.clear_session()
    gc.collect()

# =========================
# Save final results
# =========================
results_df = pd.DataFrame(all_results)
csv_path = f"results/vit_pe_results_{DATASET_NAME}.csv"
results_df.to_csv(csv_path, index=False)

print("\n✅ Final results saved to:", csv_path)
print("\nMean performance across folds:\n")

try:
    print(results_df.mean(numeric_only=True))
except Exception as e:
    print("Mean calculation skipped:", e)

✅ Loaded Hyperparameters for Raw: {'patch_size': 320, 'projection_dim': 128, 'num_heads': 4, 'transformer_layers': 2, 'ff_dim': 256, 'dropout_rate': 0.34097, 'learning_rate': 0.000192, 'batch_size': 32}
Total samples: 20792
Total subjects: 87

Fold 1/5
Train samples: 16480 | Val samples: 4312
Epoch 1/50
515/515 [==============================] - 48s 85ms/step - loss: 0.6670 - accuracy: 0.5936 - val_loss: 0.6132 - val_accuracy: 0.6667 - lr: 1.9200e-04
Epoch 2/50
515/515 [==============================] - 26s 51ms/step - loss: 0.6136 - accuracy: 0.6625 - val_loss: 0.6000 - val_accuracy: 0.6897 - lr: 1.9200e-04
Epoch 3/50
515/515 [==============================] - 24s 46ms/step - loss: 0.5783 - accuracy: 0.6968 - val_loss: 0.5929 - val_accuracy: 0.6892 - lr: 1.9200e-04
Epoch 4/50
515/515 [==============================] - 23s 44ms/step - loss: 0.5482 - accuracy: 0.7197 - val_loss: 0.5881 - val_accuracy: 0.6895 - lr: 1.9200e-04
Epoch 5/50
515/515 [==============================] - 23s 44ms

In [5]:
import os
import time
import random
import gc
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import (
    Layer, Input, Dense, LayerNormalization, MultiHeadAttention,
    Dropout, GlobalAveragePooling1D
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# =========================
# Reproducibility
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# =========================
# Dataset parameters
# =========================
EPOCHS = 50
N_SPLITS = 5
CHANNELS = 27
TIMEPOINTS = 2560
N_CLASSES = 2

DATASET_NAME = "Filtered_8_30Hz"
CSV_FILE = "dataset_metadata_filtered.csv"

# =========================
# Smart Hyperparameters Dictionary
# =========================
ALL_HYPERPARAMS = {
    "Raw": {
        'patch_size': 320, 'projection_dim': 128, 'num_heads': 4,
        'transformer_layers': 2, 'ff_dim': 256, 'dropout_rate': 0.34097,
        'learning_rate': 0.000192, 'batch_size': 32
    },
    "Filtered_8_30Hz": {
        'patch_size': 128, 'projection_dim': 128, 'num_heads': 8,
        'transformer_layers': 2, 'ff_dim': 128, 'dropout_rate': 0.36094,
        'learning_rate': 0.000173, 'batch_size': 32
    },
    "Filtered_8_13Hz": {
        'patch_size': 128, 'projection_dim': 64, 'num_heads': 4,
        'transformer_layers': 6, 'ff_dim': 128, 'dropout_rate': 0.37625,
        'learning_rate': 0.000185, 'batch_size': 16
    }
}

best_params = ALL_HYPERPARAMS[DATASET_NAME]
print(f"✅ Loaded Hyperparameters for {DATASET_NAME}: {best_params}")

# =========================
# Load metadata
# =========================
df = pd.read_csv(CSV_FILE)
print(f"Total samples: {len(df)}")
print(f"Total subjects: {df['subject'].nunique()}")

# =========================
# Data Generator (Adapted for ViT shape)
# =========================
class EEGGenerator_ViT(Sequence):
    def __init__(self, df, batch_size=32, shuffle=True):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indexes[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_df = self.df.iloc[batch_idx]

        X, y = [], []
        for _, row in batch_df.iterrows():
            data = np.load(row["file"])
            # ViT expects (TIMEPOINTS, CHANNELS) -> (2560, 27)
            # which is already the raw shape, so no transpose or newaxis is needed!
            X.append(data)
            y.append(row["label"])

        X = np.array(X, dtype=np.float32)
        y = tf.keras.utils.to_categorical(y, N_CLASSES)
        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

# =========================
# Transformer Layers
# =========================
class Patches(Layer):
    def __init__(self, patch_size, **kwargs):
        super(Patches, self).__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, signals):
        batch_size = tf.shape(signals)[0]
        # Reshape to (batch, num_patches, patch_size, channels)
        patches = tf.reshape(signals, [batch_size, -1, self.patch_size, tf.shape(signals)[-1]])
        return patches

    def get_config(self):
        config = super(Patches, self).get_config()
        config.update({"patch_size": self.patch_size})
        return config

class PatchEncoder(Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super(PatchEncoder, self).__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = Dense(units=projection_dim)
        
        pos_encoding_matrix = self.get_sinusoidal_position_embedding(num_patches, projection_dim)
        self.position_embedding = tf.Variable(
            pos_encoding_matrix, trainable=False, name="position_embedding"
        )

    def get_sinusoidal_position_embedding(self, sequence_length, embedding_dim):
        angle_rates = 1.0 / (10000 ** (tf.range(0, embedding_dim, 2, dtype=tf.float32) / float(embedding_dim)))
        positions = tf.range(sequence_length, dtype=tf.float32)[:, tf.newaxis]
        angle_rads = positions * angle_rates[tf.newaxis, :]
        sines = tf.math.sin(angle_rads)
        cosines = tf.math.cos(angle_rads)
        pos_encoding = tf.concat([sines, cosines], axis=-1)
        return pos_encoding

    def call(self, patch):
        batch_size = tf.shape(patch)[0]
        patch_height = patch.shape[-2]
        patch_width = patch.shape[-1]
        
        patch_flattened = tf.reshape(patch, [batch_size, self.num_patches, patch_height * patch_width])
        encoded = self.projection(patch_flattened)
        encoded += self.position_embedding 
        return encoded

    def get_config(self):
        config = super(PatchEncoder, self).get_config()
        config.update({"num_patches": self.num_patches, "projection_dim": self.projection_dim})
        return config

# =========================
# ViT-PE Model Builder
# =========================
def build_vit_pe(seq_len, input_dim, patch_size, projection_dim, num_heads, transformer_layers, ff_dim, dropout_rate, lr):
    assert seq_len % patch_size == 0, "Sequence length must be divisible by patch size."
    num_patches = seq_len // patch_size

    inputs = Input(shape=(seq_len, input_dim))
    patches = Patches(patch_size)(inputs)
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

    for _ in range(transformer_layers):
        x1 = LayerNormalization(epsilon=1e-6)(encoded_patches)
        attention_output = MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim, dropout=dropout_rate
        )(x1, x1)
        x2 = tf.keras.layers.Add()([attention_output, encoded_patches])
        x3 = LayerNormalization(epsilon=1e-6)(x2)
        x3 = Dense(ff_dim, activation="relu")(x3)
        x3 = Dropout(dropout_rate)(x3)
        x3 = Dense(projection_dim)(x3)
        encoded_patches = tf.keras.layers.Add()([x3, x2])

    representation = LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = GlobalAveragePooling1D()(representation)
    representation = Dropout(0.5)(representation)
    
    outputs = Dense(128, activation='relu')(representation)
    # ✅ Changed to Softmax for standard comparison with EEGNet
    outputs = Dense(N_CLASSES, activation='softmax')(outputs)

    model = Model(inputs=inputs, outputs=outputs)
    optimizer = Adam(learning_rate=lr)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# =========================
# Results storage
# =========================
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)
all_results = []

# =========================
# GroupKFold cross-validation
# =========================
gkf = GroupKFold(n_splits=N_SPLITS)
groups = df["subject"]

for fold, (train_idx, val_idx) in enumerate(gkf.split(df, df["label"], groups)):
    print(f"\n=============================")
    print(f"Fold {fold + 1}/{N_SPLITS}")
    print("=============================")

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    print(f"Train samples: {len(train_df)} | Val samples: {len(val_df)}")

    train_gen = EEGGenerator_ViT(train_df, best_params['batch_size'], shuffle=True)
    val_gen = EEGGenerator_ViT(val_df, best_params['batch_size'], shuffle=False)

    model = build_vit_pe(
        seq_len=TIMEPOINTS,
        input_dim=CHANNELS,
        patch_size=best_params['patch_size'],
        projection_dim=best_params['projection_dim'],
        num_heads=best_params['num_heads'],
        transformer_layers=best_params['transformer_layers'],
        ff_dim=best_params['ff_dim'],
        dropout_rate=best_params['dropout_rate'],
        lr=best_params['learning_rate']
    )

    checkpoint_path = f"models/vit_pe_{DATASET_NAME}_fold{fold + 1}.h5"
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        ModelCheckpoint(checkpoint_path, monitor="val_loss", save_best_only=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6)
    ]

    start_time = time.time()
    history = model.fit(train_gen, validation_data=val_gen,
                        epochs=EPOCHS, callbacks=callbacks, verbose=1)
    end_time = time.time()
    print(f"Training time: {end_time - start_time:.2f} seconds")

    # =========================
    # Evaluation
    # =========================
    y_true, y_pred, y_prob = [], [], []
    for X_batch, y_batch in val_gen:
        preds = model.predict(X_batch, verbose=0)
        y_true.extend(np.argmax(y_batch, axis=1))
        y_pred.extend(np.argmax(preds, axis=1))
        y_prob.extend(preds[:, 1])

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc = roc_auc_score(y_true, y_prob)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp)

    print(f"Accuracy: {acc:.3f} | F1: {f1:.3f} | ROC-AUC: {roc:.3f}")

    fold_result = {
        "dataset": DATASET_NAME,
        "fold": fold + 1,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "roc_auc": roc,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn
    }
    all_results.append(fold_result)

    del model
    tf.keras.backend.clear_session()
    gc.collect()

# =========================
# Save final results
# =========================
results_df = pd.DataFrame(all_results)
csv_path = f"results/vit_pe_results_{DATASET_NAME}.csv"
results_df.to_csv(csv_path, index=False)

print("\n✅ Final results saved to:", csv_path)
print("\nMean performance across folds:\n")

try:
    print(results_df.mean(numeric_only=True))
except Exception as e:
    print("Mean calculation skipped:", e)

✅ Loaded Hyperparameters for Filtered_8_30Hz: {'patch_size': 128, 'projection_dim': 128, 'num_heads': 8, 'transformer_layers': 2, 'ff_dim': 128, 'dropout_rate': 0.36094, 'learning_rate': 0.000173, 'batch_size': 32}
Total samples: 20792
Total subjects: 87

Fold 1/5
Train samples: 16480 | Val samples: 4312
Epoch 1/50
515/515 [==============================] - 42s 77ms/step - loss: 0.7049 - accuracy: 0.5110 - val_loss: 0.6895 - val_accuracy: 0.5332 - lr: 1.7300e-04
Epoch 2/50
515/515 [==============================] - 23s 44ms/step - loss: 0.6921 - accuracy: 0.5336 - val_loss: 0.6871 - val_accuracy: 0.5369 - lr: 1.7300e-04
Epoch 3/50
515/515 [==============================] - 20s 38ms/step - loss: 0.6874 - accuracy: 0.5454 - val_loss: 0.6843 - val_accuracy: 0.5582 - lr: 1.7300e-04
Epoch 4/50
515/515 [==============================] - 20s 38ms/step - loss: 0.6709 - accuracy: 0.5839 - val_loss: 0.6619 - val_accuracy: 0.5969 - lr: 1.7300e-04
Epoch 5/50
515/515 [==============================

In [6]:
import os
import time
import random
import gc
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import (
    Layer, Input, Dense, LayerNormalization, MultiHeadAttention,
    Dropout, GlobalAveragePooling1D
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# =========================
# Reproducibility
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# =========================
# Dataset parameters
# =========================
EPOCHS = 50
N_SPLITS = 5
CHANNELS = 27
TIMEPOINTS = 2560
N_CLASSES = 2

DATASET_NAME = "Filtered_8_13Hz"
CSV_FILE = "dataset_metadata_8_13Hz.csv"

# =========================
# Smart Hyperparameters Dictionary
# =========================
ALL_HYPERPARAMS = {
    "Raw": {
        'patch_size': 320, 'projection_dim': 128, 'num_heads': 4,
        'transformer_layers': 2, 'ff_dim': 256, 'dropout_rate': 0.34097,
        'learning_rate': 0.000192, 'batch_size': 32
    },
    "Filtered_8_30Hz": {
        'patch_size': 128, 'projection_dim': 128, 'num_heads': 8,
        'transformer_layers': 2, 'ff_dim': 128, 'dropout_rate': 0.36094,
        'learning_rate': 0.000173, 'batch_size': 32
    },
    "Filtered_8_13Hz": {
        'patch_size': 128, 'projection_dim': 64, 'num_heads': 4,
        'transformer_layers': 6, 'ff_dim': 128, 'dropout_rate': 0.37625,
        'learning_rate': 0.000185, 'batch_size': 16
    }
}

best_params = ALL_HYPERPARAMS[DATASET_NAME]
print(f"✅ Loaded Hyperparameters for {DATASET_NAME}: {best_params}")

# =========================
# Load metadata
# =========================
df = pd.read_csv(CSV_FILE)
print(f"Total samples: {len(df)}")
print(f"Total subjects: {df['subject'].nunique()}")

# =========================
# Data Generator (Adapted for ViT shape)
# =========================
class EEGGenerator_ViT(Sequence):
    def __init__(self, df, batch_size=32, shuffle=True):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.df))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indexes[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_df = self.df.iloc[batch_idx]

        X, y = [], []
        for _, row in batch_df.iterrows():
            data = np.load(row["file"])
            # ViT expects (TIMEPOINTS, CHANNELS) -> (2560, 27)
            # which is already the raw shape, so no transpose or newaxis is needed!
            X.append(data)
            y.append(row["label"])

        X = np.array(X, dtype=np.float32)
        y = tf.keras.utils.to_categorical(y, N_CLASSES)
        return X, y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

# =========================
# Transformer Layers
# =========================
class Patches(Layer):
    def __init__(self, patch_size, **kwargs):
        super(Patches, self).__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, signals):
        batch_size = tf.shape(signals)[0]
        # Reshape to (batch, num_patches, patch_size, channels)
        patches = tf.reshape(signals, [batch_size, -1, self.patch_size, tf.shape(signals)[-1]])
        return patches

    def get_config(self):
        config = super(Patches, self).get_config()
        config.update({"patch_size": self.patch_size})
        return config

class PatchEncoder(Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super(PatchEncoder, self).__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = Dense(units=projection_dim)
        
        pos_encoding_matrix = self.get_sinusoidal_position_embedding(num_patches, projection_dim)
        self.position_embedding = tf.Variable(
            pos_encoding_matrix, trainable=False, name="position_embedding"
        )

    def get_sinusoidal_position_embedding(self, sequence_length, embedding_dim):
        angle_rates = 1.0 / (10000 ** (tf.range(0, embedding_dim, 2, dtype=tf.float32) / float(embedding_dim)))
        positions = tf.range(sequence_length, dtype=tf.float32)[:, tf.newaxis]
        angle_rads = positions * angle_rates[tf.newaxis, :]
        sines = tf.math.sin(angle_rads)
        cosines = tf.math.cos(angle_rads)
        pos_encoding = tf.concat([sines, cosines], axis=-1)
        return pos_encoding

    def call(self, patch):
        batch_size = tf.shape(patch)[0]
        patch_height = patch.shape[-2]
        patch_width = patch.shape[-1]
        
        patch_flattened = tf.reshape(patch, [batch_size, self.num_patches, patch_height * patch_width])
        encoded = self.projection(patch_flattened)
        encoded += self.position_embedding 
        return encoded

    def get_config(self):
        config = super(PatchEncoder, self).get_config()
        config.update({"num_patches": self.num_patches, "projection_dim": self.projection_dim})
        return config

# =========================
# ViT-PE Model Builder
# =========================
def build_vit_pe(seq_len, input_dim, patch_size, projection_dim, num_heads, transformer_layers, ff_dim, dropout_rate, lr):
    assert seq_len % patch_size == 0, "Sequence length must be divisible by patch size."
    num_patches = seq_len // patch_size

    inputs = Input(shape=(seq_len, input_dim))
    patches = Patches(patch_size)(inputs)
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

    for _ in range(transformer_layers):
        x1 = LayerNormalization(epsilon=1e-6)(encoded_patches)
        attention_output = MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim, dropout=dropout_rate
        )(x1, x1)
        x2 = tf.keras.layers.Add()([attention_output, encoded_patches])
        x3 = LayerNormalization(epsilon=1e-6)(x2)
        x3 = Dense(ff_dim, activation="relu")(x3)
        x3 = Dropout(dropout_rate)(x3)
        x3 = Dense(projection_dim)(x3)
        encoded_patches = tf.keras.layers.Add()([x3, x2])

    representation = LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = GlobalAveragePooling1D()(representation)
    representation = Dropout(0.5)(representation)
    
    outputs = Dense(128, activation='relu')(representation)
    # ✅ Changed to Softmax for standard comparison with EEGNet
    outputs = Dense(N_CLASSES, activation='softmax')(outputs)

    model = Model(inputs=inputs, outputs=outputs)
    optimizer = Adam(learning_rate=lr)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# =========================
# Results storage
# =========================
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)
all_results = []

# =========================
# GroupKFold cross-validation
# =========================
gkf = GroupKFold(n_splits=N_SPLITS)
groups = df["subject"]

for fold, (train_idx, val_idx) in enumerate(gkf.split(df, df["label"], groups)):
    print(f"\n=============================")
    print(f"Fold {fold + 1}/{N_SPLITS}")
    print("=============================")

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    print(f"Train samples: {len(train_df)} | Val samples: {len(val_df)}")

    train_gen = EEGGenerator_ViT(train_df, best_params['batch_size'], shuffle=True)
    val_gen = EEGGenerator_ViT(val_df, best_params['batch_size'], shuffle=False)

    model = build_vit_pe(
        seq_len=TIMEPOINTS,
        input_dim=CHANNELS,
        patch_size=best_params['patch_size'],
        projection_dim=best_params['projection_dim'],
        num_heads=best_params['num_heads'],
        transformer_layers=best_params['transformer_layers'],
        ff_dim=best_params['ff_dim'],
        dropout_rate=best_params['dropout_rate'],
        lr=best_params['learning_rate']
    )

    checkpoint_path = f"models/vit_pe_{DATASET_NAME}_fold{fold + 1}.h5"
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        ModelCheckpoint(checkpoint_path, monitor="val_loss", save_best_only=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6)
    ]

    start_time = time.time()
    history = model.fit(train_gen, validation_data=val_gen,
                        epochs=EPOCHS, callbacks=callbacks, verbose=1)
    end_time = time.time()
    print(f"Training time: {end_time - start_time:.2f} seconds")

    # =========================
    # Evaluation
    # =========================
    y_true, y_pred, y_prob = [], [], []
    for X_batch, y_batch in val_gen:
        preds = model.predict(X_batch, verbose=0)
        y_true.extend(np.argmax(y_batch, axis=1))
        y_pred.extend(np.argmax(preds, axis=1))
        y_prob.extend(preds[:, 1])

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc = roc_auc_score(y_true, y_prob)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp)

    print(f"Accuracy: {acc:.3f} | F1: {f1:.3f} | ROC-AUC: {roc:.3f}")

    fold_result = {
        "dataset": DATASET_NAME,
        "fold": fold + 1,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "roc_auc": roc,
        "tp": tp, "tn": tn, "fp": fp, "fn": fn
    }
    all_results.append(fold_result)

    del model
    tf.keras.backend.clear_session()
    gc.collect()

# =========================
# Save final results
# =========================
results_df = pd.DataFrame(all_results)
csv_path = f"results/vit_pe_results_{DATASET_NAME}.csv"
results_df.to_csv(csv_path, index=False)

print("\n✅ Final results saved to:", csv_path)
print("\nMean performance across folds:\n")

try:
    print(results_df.mean(numeric_only=True))
except Exception as e:
    print("Mean calculation skipped:", e)

✅ Loaded Hyperparameters for Filtered_8_13Hz: {'patch_size': 128, 'projection_dim': 64, 'num_heads': 4, 'transformer_layers': 6, 'ff_dim': 128, 'dropout_rate': 0.37625, 'learning_rate': 0.000185, 'batch_size': 16}
Total samples: 20792
Total subjects: 87

Fold 1/5
Train samples: 16480 | Val samples: 4312
Epoch 1/50
1030/1030 [==============================] - 69s 62ms/step - loss: 0.7010 - accuracy: 0.5115 - val_loss: 0.6899 - val_accuracy: 0.5311 - lr: 1.8500e-04
Epoch 2/50
1030/1030 [==============================] - 43s 41ms/step - loss: 0.6913 - accuracy: 0.5311 - val_loss: 0.6848 - val_accuracy: 0.5571 - lr: 1.8500e-04
Epoch 3/50
1030/1030 [==============================] - 43s 42ms/step - loss: 0.6847 - accuracy: 0.5535 - val_loss: 0.6790 - val_accuracy: 0.5700 - lr: 1.8500e-04
Epoch 4/50
1030/1030 [==============================] - 43s 41ms/step - loss: 0.6775 - accuracy: 0.5760 - val_loss: 0.6701 - val_accuracy: 0.5853 - lr: 1.8500e-04
Epoch 5/50
1030/1030 [=====================